# RQ1_new — analysis

Reads from `clean_data/<model>.csv`, built by `build_clean_dataset.py` — **only from rounds verified round-by-round to not have the silent-no-contention bug found 2026-08-05**, never from everything collected so far. See `clean_data/MANIFEST.json` for exactly which rounds went in and why; `model2` and `model3` (sib_cfs) currently have nothing valid pooled yet — every existing round of both predates the fix and showed the bug. Re-run `build_clean_dataset.py` after collecting/verifying new rounds.

**Definitions** (raw jobs.csv columns → derived): `D`=period, `Q`=round(U·P), `bound`=max(1, 2(P−Q)), `alpha`=C/Q, `delta`=(R−C)/bound, `R/D`=R/P. Raw (non-normalized) counterparts shown alongside: `C_cputime_us` itself, and `R_wall_us − C_cputime_us` (added latency, µs).

In [ ]:
import analysis_lib as al
import pandas as pd
pd.set_option("display.width", 140)

MODELS = ["model1", "model3-w2", "model3-w3", "model3-w4"]  # model2, model3 excluded: nothing valid yet
data = {m: al.load_model(m) for m in MODELS}

## Section A — pooled dataset (all U together)

### A.1 Basic descriptives

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    print(f"=== {m} ===")
    display(al.descriptives_table(data[m]))

### A.2 Distribution of R and C over U
Boxplots, not a single collapsed histogram — R and C aren't comparable across different U cells (different Q/period), so the shape needs to be shown *per U*, not pooled across it.

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    al.distribution_over_u(data[m], "R_wall_us", "R (\u00b5s)", f"{m}: R distribution")
    al.distribution_over_u(data[m], "C_cputime_us", "C (\u00b5s)", f"{m}: C distribution")

### A.3 Budget overrun (alpha > 1)
How often execution ALONE exceeds the reservation's own budget — independent of whether the job also missed its deadline.

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    print(f"=== {m} ===")
    display(al.budget_overrun_table(data[m]))

### A.4 Round-to-round stability (QA check, keep this)
Straight from the included rounds' own raw data, not the pooled file — this is the exact check that caught the coin-flip bug. A round whose numbers don't match its siblings at the same U is the tell; pooling alone hides it. Re-check this whenever a new round is added to `clean_data`.

In [ ]:
for m in MODELS:
    t = al.round_stability_table(m)
    if t.empty: continue
    print(f"=== {m} ===")
    display(t)

### A.5 Burst of missed jobs (longest consecutive run)
Temporal statistic — computed per round on that round's own job-order sequence, then MAXED across included rounds. Never concatenate raw sequences across rounds for this one (see `analysis_lib.longest_miss_run_table` docstring).

In [ ]:
for m in MODELS:
    t = al.longest_miss_run_table(m)
    if t.empty: continue
    print(f"=== {m} ===")
    display(t)

### A.7 Mid-job preemption: how many, how long
How many jobs were actually preempted mid-execution, and for how long -- the raw signal behind A.5's miss-run bursts. A tight duration distribution across many affected jobs (small std relative to the mean) is the signature of a recurring external event with a near-fixed cost -- worth comparing `mean_preempt_us` against any fixed-intensity co-runner's own budget (`co_runners.competitor.u * P`) in the model's config, which is exactly what explained the model3-w4 soft/U0.6 burst (see thesis notes).

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    print(f"=== {m} ===")
    display(al.preemption_table(data[m]))

### A.6 p99 / p999 safe margins
The empirical "how far can you provision before this breaks the deadline" answer — largest U at which p99/p999 of R/D stays under 1.

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    print(f"=== {m} ===")
    display(al.safe_margin_table(data[m]))

## Section B — per utilization (U)

### B.1 alpha (C/Q, normalized) and raw C, vs U
Shaded band = p10–p90 (stability at that U); dashed = p99. Reference line at alpha=1 (=1 line: guarantee-hold boundary for the normalized panel; raw-C panel has no single universal reference since Q itself changes with U).

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    al.plot_vs_u(data[m], "alpha", "alpha = C/Q", f"{m}: alpha vs U", hline=1, hline_label="alpha=1 (budget)")
    al.plot_vs_u(data[m], "C_cputime_us", "C (\u00b5s, raw)", f"{m}: raw C vs U")

### B.2 delta (normalized) and raw added latency (R−C), vs U

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    al.plot_vs_u(data[m], "delta", "delta = (R−C)/bound", f"{m}: delta vs U", hline=1, hline_label="delta=1")
    al.plot_vs_u(data[m], "added_latency_us", "R−C (\u00b5s, raw)", f"{m}: raw added latency vs U")

### B.3 Deadline miss rate vs U

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    al.miss_rate_vs_u(data[m])

### B.4 p50 / p99 of R and C together, vs U
Same unit (µs) so one y-axis; color = R vs C, linestyle = p50 vs p99. Never a second y-axis for a same-unit second series.

In [ ]:
for m in MODELS:
    if data[m].empty: continue
    al.rc_percentiles_vs_u(data[m])